# Task 2: Parser Service CPG

## Mục tiêu

Task này xây dựng Parser Service xử lý từng file Python riêng lẻ và sinh event cho Code Property Graph (CPG). Service sử dụng module chuẩn `ast` của Python thông qua các classes trong `src/parsing/` để lấy AST node, quan hệ cha-con trong AST, call edge, CFG và DFG.

Notebook này gọi CLI chính thức của dự án để parse mã nguồn ở chế độ dry-run, sinh ra các file JSONL cục bộ. Sau đó notebook kiểm tra tính nhất quán cấu trúc event với JSON Schema thực tế, kiểm tra tính deterministic của Stable IDs, cơ chế skip file không thay đổi, và cơ chế bắt lỗi cú pháp.


## Thiết Kế Parser Service

Sau khi refactor, Parser Service được triển khai theo kiến trúc mới dưới `src/`. Cấu trúc chính:

| File | Vai trò |
|---|---|
| `parser.py` | CLI entrypoint, duyệt danh sách file và điều phối parse/write |
| `cpg_parser.py` | Logic tạo AST/CFG/DFG/Call event bằng `ast` |
| `event_writer.py` | Ghi JSONL khi dry-run hoặc publish Kafka khi chạy thật |
| `stable_id.py` | Hàm hash tạo định danh ổn định |
| `topics.py` | Tên topic Kafka cho từng nhóm event |
| `schemas/*.schema.json` | JSON Schema mô tả cấu trúc event |

Ở chế độ `--dry-run`, service ghi mỗi nhóm event thành một file JSONL để kiểm tra nhanh trước khi bật Kafka.

In [1]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from collections import Counter
import jsonschema

# Resolve PROJECT_ROOT
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

# Isolated paths for Task 2 demonstration
TASK2_STATE_DB = PROJECT_ROOT / "workspace/state/book_task2_parser_state.sqlite3"
TASK2_OUT_DIR = PROJECT_ROOT / "workspace/tmp/book-task2-output"
SCHEMA_DIR = PROJECT_ROOT / "schemas"
SOURCE_REPOSITORY = PROJECT_ROOT / "workspace/source/transformers-pr-agent"

# Reset isolated Task 2 demonstration state and dirs
if TASK2_STATE_DB.exists():
    TASK2_STATE_DB.unlink()
if TASK2_OUT_DIR.exists():
    shutil.rmtree(TASK2_OUT_DIR)
TASK2_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Isolated state DB:", TASK2_STATE_DB)
print("Isolated output dir:", TASK2_OUT_DIR)
print("Schemas directory:", SCHEMA_DIR)
print("SOURCE_REPOSITORY:", SOURCE_REPOSITORY)


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming
Isolated state DB: /home/phat/AI_Project/lab04-cpg-streaming/workspace/state/book_task2_parser_state.sqlite3
Isolated output dir: /home/phat/AI_Project/lab04-cpg-streaming/workspace/tmp/book-task2-output
Schemas directory: /home/phat/AI_Project/lab04-cpg-streaming/schemas
SOURCE_REPOSITORY: /home/phat/AI_Project/lab04-cpg-streaming/workspace/source/transformers-pr-agent


## Chạy Parser Service Ở Chế Độ Dry Run

Notebook parse 3 file Python đầu tiên trong `src/`. Giới hạn nhỏ giúp quá trình chạy nhanh nhưng vẫn tạo đủ node, edge và metadata để kiểm tra schema event. Khi chạy pipeline đầy đủ, có thể bỏ `--limit`.

In [2]:
# Initial parse run on 3 files using the official CLI
env = {**os.environ, "PARSER_STATE_DB": str(TASK2_STATE_DB)}

result = subprocess.run(
    [
        "uv", "run", "lab04", "parse-repository",
        "--scope", "smoke",
        "--limit", "3",
        "--dry-run",
        "--clean-output",
        "--out-dir", str(TASK2_OUT_DIR / "initial"),
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print("CLI command executed successfully!")
print(result.stdout)


CLI command executed successfully!
Parsing repository (scope=smoke, limit=3, dry_run=True)...
Repository run completed. Summary:
{'discovered': 2779, 'eligible': 3, 'processed': 3, 'skipped_unchanged': 0, 'failed': 0, 'node_events': 4699, 'edge_events': 5453, 'metadata_events': 3, 'error_events': 0, 'duration_ms': 100476}



## Thống Kê Event Đã Sinh

Mỗi dòng trong file JSONL tương ứng với một message sẽ được gửi vào Kafka khi chạy thật. Việc đếm số dòng giúp kiểm tra nhanh parser có sinh đủ bốn nhóm event hay không.

In [3]:
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open(encoding="utf-8") as handle:
        return sum(1 for _ in handle)

initial_out = TASK2_OUT_DIR / "initial"
counts = {
    name: count_jsonl(initial_out / f"{name}.jsonl")
    for name in ["nodes", "edges", "metadata", "errors"]
}

for name, count in counts.items():
    print(f"{name}: {count}")


nodes: 4699
edges: 5453
metadata: 3
errors: 0


## Mẫu Node Event

Node event biểu diễn một node trong AST hoặc một call target tổng hợp. Trường `id` được tạo bằng hash ổn định từ file, nội dung file và vị trí AST, giúp cùng một input tạo lại cùng một định danh.

In [4]:
with (initial_out / "nodes.jsonl").open(encoding="utf-8") as handle:
    sample_node = json.loads(next(handle))

print(json.dumps(sample_node, indent=2, ensure_ascii=False))


{
  "schema_version": "1.0",
  "event_id": "faec7e9100f5da0a616231c392a74a95ccce855e68772cc65f37f5743132d938",
  "event_type": "NODE_UPSERT",
  "event_time": "2026-07-21T01:05:16Z",
  "repository_id": "huggingface/transformers-pr-agent",
  "commit_sha": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_id": "526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc",
  "file_path": "src/transformers/__init__.py",
  "content_hash": "995c4799e2dbed014814fd85485e04a46b53bb4c74512f97edb0502d60917e65",
  "parser_version": "1.0.0",
  "node": {
    "node_id": "6e65fb21b5c7c88bc6d1103d9526384516bf980081cb166075fb31d1e3fac364",
    "node_type": "Module",
    "name": "Module",
    "qualified_name": "Module",
    "ast_path": "Module",
    "line_start": null,
    "column_start": null,
    "line_end": null,
    "column_end": null,
    "properties": {}
  }
}


## Mẫu Edge Event

Edge event mô tả quan hệ giữa hai node. Trong phiên bản hiện tại, service sinh bốn loại cạnh:

| Edge type | Ý nghĩa |
|---|---|
| `AST_CHILD` | Quan hệ cha-con trong AST |
| `CFG_NEXT` | Statement kế tiếp trong cùng block |
| `DFG_REACHES` | Lần gán biến gần nhất đi tới lần đọc biến |
| `CALLS` | Một `ast.Call` gọi tới tên hàm/phương thức |

In [5]:
edge_types = Counter()
first_edge = None
with (initial_out / "edges.jsonl").open(encoding="utf-8") as handle:
    for line in handle:
        event = json.loads(line)
        first_edge = first_edge or event
        edge_types[event["edge"]["edge_type"]] += 1

print("Edge types:")
for edge_type, count in edge_types.most_common():
    print(f"{edge_type}: {count}")

print("\nSample edge:")
print(json.dumps(first_edge, indent=2, ensure_ascii=False))


Edge types:
AST_CHILD: 4461
CFG_NEXT: 520
DFG_DEF_USE: 264
CALLS: 156
CFG_RETURN: 26
CFG_TRUE: 10
CFG_FALSE: 10
CFG_LOOP_BODY: 2
CFG_LOOP_EXIT: 2
CFG_LOOP_BACK: 2

Sample edge:
{
  "schema_version": "1.0",
  "event_id": "bcf862d6e486c2a299b9f3b4bc6d37710066c944c58571c2081f0c05c3269970",
  "event_type": "EDGE_UPSERT",
  "event_time": "2026-07-21T01:05:16Z",
  "repository_id": "huggingface/transformers-pr-agent",
  "commit_sha": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_id": "526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc",
  "file_path": "src/transformers/__init__.py",
  "content_hash": "995c4799e2dbed014814fd85485e04a46b53bb4c74512f97edb0502d60917e65",
  "parser_version": "1.0.0",
  "edge": {
    "edge_id": "2fc10e5bb915d017535d742086e36ca1de7cf8eaecef0a5cc1b8bc6f3c1a8448",
    "source_id": "6e65fb21b5c7c88bc6d1103d9526384516bf980081cb166075fb31d1e3fac364",
    "target_id": "ba00e64dbb110261ef4efdb669e5945d8c9d5e439e7c44c3bbe06ed18201cfb3",
    "edge_type": 

## Mẫu Metadata Event

Metadata event phục vụ pipeline MongoDB ở task sau. Event này lưu commit, đường dẫn file, hash nội dung, kích thước file, số dòng và parser đã sử dụng.

In [6]:
with (initial_out / "metadata.jsonl").open(encoding="utf-8") as handle:
    sample_metadata = json.loads(next(handle))

print(json.dumps(sample_metadata, indent=2, ensure_ascii=False))


{
  "schema_version": "1.0",
  "event_id": "9d47ada77619830e7a32b48a8de3b7ff1a1c8885ffa5d6fc6a4e553aa72aa889",
  "event_type": "FILE_METADATA_UPSERT",
  "event_time": "2026-07-21T01:05:16Z",
  "repository_id": "huggingface/transformers-pr-agent",
  "commit_sha": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_id": "526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc",
  "file_path": "src/transformers/__init__.py",
  "content_hash": "995c4799e2dbed014814fd85485e04a46b53bb4c74512f97edb0502d60917e65",
  "parser_version": "1.0.0",
  "metadata": {
    "size_bytes": 41203,
    "line_count": 863,
    "function_count": 4,
    "class_count": 0,
    "import_count": 308,
    "node_count": 1971,
    "edge_count": 2417,
    "parse_duration_ms": 46,
    "parse_status": "SUCCESS",
    "parser": "python.ast"
  }
}


## Lệnh Chạy Với Kafka

Khi Kafka broker đã chạy, bỏ `--dry-run` để gửi event thật vào Kafka:

```powershell
python scripts/run_parser.py --repo transformers-pr-agent --source-root transformers-pr-agent/src --bootstrap-servers localhost:9092
```

Nếu muốn thử trên một số file trước:

```powershell
python scripts/run_parser.py --repo transformers-pr-agent --source-root transformers-pr-agent/src --limit 3 --bootstrap-servers localhost:9092
```

In [7]:
# Schema validation helper using root schemas
def validate_event_with_schema(event: dict, schema_name: str):
    schema_path = SCHEMA_DIR / schema_name
    with open(schema_path, "r", encoding="utf-8") as f:
        schema = json.load(f)
    jsonschema.Draft202012Validator(schema).validate(event)

# Validate all nodes, edges, and metadata
event_types_checked = set()

# Check Node schemas
with (initial_out / "nodes.jsonl").open(encoding="utf-8") as f:
    for line in f:
        event = json.loads(line)
        validate_event_with_schema(event, "node-event.schema.json")
        event_types_checked.add(event["event_type"])
        
# Check Edge schemas
with (initial_out / "edges.jsonl").open(encoding="utf-8") as f:
    for line in f:
        event = json.loads(line)
        validate_event_with_schema(event, "edge-event.schema.json")
        event_types_checked.add(event["event_type"])
        
# Check Metadata schemas
with (initial_out / "metadata.jsonl").open(encoding="utf-8") as f:
    for line in f:
        event = json.loads(line)
        validate_event_with_schema(event, "metadata-event.schema.json")
        event_types_checked.add(event["event_type"])

print("JSON Schema validation using root schemas directory PASSED successfully!")
print("Validated event types:", list(event_types_checked))


JSON Schema validation using root schemas directory PASSED successfully!
Validated event types: ['EDGE_UPSERT', 'FILE_METADATA_UPSERT', 'NODE_UPSERT']


In [8]:
# Stable IDs and content hashes determinism verification A/B
state_a = TASK2_OUT_DIR / "initial"
state_b = TASK2_OUT_DIR / "run_b"
state_b_db = PROJECT_ROOT / "workspace/state/book_task2_run_b.sqlite3"
if state_b_db.exists():
    state_b_db.unlink()
    
env_b = {**os.environ, "PARSER_STATE_DB": str(state_b_db)}

subprocess.run(
    [
        "uv", "run", "lab04", "parse-repository",
        "--scope", "smoke",
        "--limit", "3",
        "--dry-run",
        "--clean-output",
        "--out-dir", str(state_b),
    ],
    check=True,
    capture_output=True,
    cwd=str(PROJECT_ROOT),
    env=env_b
)

def get_ids_and_hashes(out_dir_path: Path):
    node_ids = set()
    edge_ids = set()
    file_ids = set()
    hashes = set()
    
    with (out_dir_path / "nodes.jsonl").open(encoding="utf-8") as f:
        for line in f:
            evt = json.loads(line)
            node_ids.add(evt["node"]["node_id"])
    with (out_dir_path / "edges.jsonl").open(encoding="utf-8") as f:
        for line in f:
            evt = json.loads(line)
            edge_ids.add(evt["edge"]["edge_id"])
    with (out_dir_path / "metadata.jsonl").open(encoding="utf-8") as f:
        for line in f:
            evt = json.loads(line)
            file_ids.add(evt["file_id"])
            hashes.add(evt["content_hash"])
    return node_ids, edge_ids, file_ids, hashes

node_ids_a, edge_ids_a, file_ids_a, hashes_a = get_ids_and_hashes(state_a)
node_ids_b, edge_ids_b, file_ids_b, hashes_b = get_ids_and_hashes(state_b)

assert node_ids_a == node_ids_b, "Node IDs are not deterministic"
assert edge_ids_a == edge_ids_b, "Edge IDs are not deterministic"
assert file_ids_a == file_ids_b, "File IDs are not deterministic"
assert hashes_a == hashes_b, "Content hashes are not deterministic"

print("Stable IDs and content hashes determinism verification A/B PASSED successfully!")


Stable IDs and content hashes determinism verification A/B PASSED successfully!


In [9]:
# Incremental skip verification (re-parsing same repo with same state DB)
env_skip = {**os.environ, "PARSER_STATE_DB": str(TASK2_STATE_DB)}
skip_out = TASK2_OUT_DIR / "unchanged"

result_skip = subprocess.run(
    [
        "uv", "run", "lab04", "parse-repository",
        "--scope", "smoke",
        "--limit", "3",
        "--dry-run",
        "--clean-output",
        "--out-dir", str(skip_out),
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env_skip
)

print(result_skip.stdout)
assert count_jsonl(skip_out / "nodes.jsonl") == 0, "No new nodes should be generated"
assert count_jsonl(skip_out / "edges.jsonl") == 0, "No new edges should be generated"
assert count_jsonl(skip_out / "metadata.jsonl") == 0, "No new metadata should be generated"

print("Incremental unchanged-file skip verification PASSED successfully!")


Parsing repository (scope=smoke, limit=3, dry_run=True)...
Repository run completed. Summary:
{'discovered': 2779, 'eligible': 3, 'processed': 0, 'skipped_unchanged': 3, 'failed': 0, 'node_events': 0, 'edge_events': 0, 'metadata_events': 0, 'error_events': 0, 'duration_ms': 486}

Incremental unchanged-file skip verification PASSED successfully!


In [10]:
# Parser error verification using broken syntax fixture
error_db = PROJECT_ROOT / "workspace/state/book_task2_error_state.sqlite3"
if error_db.exists():
    error_db.unlink()
    
# Copy fixture to target repo to comply with CLI path resolution logic
target_fixture_dir = SOURCE_REPOSITORY / "tests/fixtures"
target_fixture_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(PROJECT_ROOT / "tests/fixtures/broken_syntax.py", target_fixture_dir / "broken_syntax.py")

env_err = {**os.environ, "PARSER_STATE_DB": str(error_db)}
err_out = TASK2_OUT_DIR / "error-case"

result_err = subprocess.run(
    [
        "uv", "run", "lab04", "parse-file",
        "--file", "tests/fixtures/broken_syntax.py",
        "--dry-run",
        "--clean-output",
        "--out-dir", str(err_out),
    ],
    check=False,  # Parse error is expected to exit with non-zero code
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env_err
)

# Cleanup copied fixture immediately
if (target_fixture_dir / "broken_syntax.py").exists():
    (target_fixture_dir / "broken_syntax.py").unlink()

print("Exit code:", result_err.returncode)
print(result_err.stdout)

# Verify error JSONL exists and conforms to schema
assert (err_out / "errors.jsonl").exists(), "errors.jsonl must be created"
with (err_out / "errors.jsonl").open(encoding="utf-8") as f:
    err_event = json.loads(next(f))
    
print("\nSample Error Event:")
print(json.dumps(err_event, indent=2, ensure_ascii=False))

validate_event_with_schema(err_event, "error-event.schema.json")
print("Parser error event validation PASSED successfully!")


Exit code: 1
Parsing single file: tests/fixtures/broken_syntax.py
File processed. Status: FAILED, content_hash: c66248af053766c7e02b9681da497cc4e2e6041914782231a74d41e60929eea9
Error details: SyntaxError in tests/fixtures/broken_syntax.py: invalid syntax at line 3, col 10


Sample Error Event:
{
  "schema_version": "1.0",
  "event_id": "d0a97b05545a7a843a17f8e16b20ffb406533dafc74d94fbff6252c2cd5fa35c",
  "event_type": "PARSER_ERROR",
  "event_time": "2026-07-21T01:08:40Z",
  "repository_id": "huggingface/transformers-pr-agent",
  "commit_sha": "458c957fa1e8851825cd799f5d030876f0644194",
  "file_id": "7481c9ee519bfe2994cb9b115331afd556b430686776493679fb0801ae77c6fc",
  "file_path": "tests/fixtures/broken_syntax.py",
  "content_hash": "c66248af053766c7e02b9681da497cc4e2e6041914782231a74d41e60929eea9",
  "parser_version": "1.0.0",
  "error": {
    "error_type": "SyntaxError",
    "message": "SyntaxError in tests/fixtures/broken_syntax.py: invalid syntax at line 3, col 10",
    "line": nul

## Kết Quả

Với 3 file đầu tiên trong `src/`, Parser Service sinh được các file `nodes.jsonl`, `edges.jsonl`, `metadata.jsonl` và `errors.jsonl` trong `outputs/parser-output-demo/`. Số lượng event được tính trực tiếp từ output ở cell thống kê phía trên, tránh hard-code số liệu khi repository hoặc parser thay đổi.

Các event này là đầu vào cho Task 3 về topic/schema Kafka và cho quá trình ingest vào Neo4j/MongoDB ở các task sau.

## Reflection

Parser Service đã chạy được ở chế độ dry-run nên có thể kiểm tra event trước khi phụ thuộc vào Kafka. Cách dùng `ast` phù hợp với phạm vi lab vì không cần parser ngoài, dễ giải thích và chạy ổn định trên Python source code.

Giới hạn hiện tại là CFG và DFG mới ở mức xấp xỉ: CFG nối các statement liên tiếp trong cùng block, còn DFG nối lần gán biến gần nhất tới lần đọc biến sau đó. Mức này đủ để minh họa CPG và thiết kế pipeline streaming, nhưng chưa thay thế được phân tích chương trình chuyên sâu trong hệ thống production.